# Compare DBs With Respect to Deceleration, and Other Player Information

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

In this file we want to compare DBs based on their deceleration ability (from `decel_score`). We will create a table summarizing these scores and other important player information. These tables will help us identify undervalues, overvalued, and properly valued players with respect to minimizing separation.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
# pio.renderers.default = "png"

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Players Data

In [ ]:
# ======================================
# Load Players Data
# ======================================

players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
    # -------------------------------
    # Change Missing draft_overall_pick To "UDFA" 
    # -------------------------------
    .with_columns(
        draft_overall_pick = pl.col("draft_overall_pick").fill_null("UDFA")
    )
    # -------------------------------
    # Only Keep WRs
    # -------------------------------
    .filter(
        pl.col("nfl_position").is_in(["CB", "FS", "SS"])
    )
    # -------------------------------
    # Only Keep Needed Variables
    # -------------------------------
    .select(
        "nfl_id", 
        "display_name", 
        "nfl_position", 
        "draft_round", 
        "draft_overall_pick"
    )
)

print("Players Data:")
print(f"    - Dimensions: {players_df.shape}")
print(f"    - Number of Players: {players_df.select("nfl_id").n_unique()}")
players_df

<br><br/>
#### Combine Features Data

In [ ]:
# =========================================
# Load Combine Data
# =========================================

# -----------------------------------------
# Results Features
# -----------------------------------------

combine_results_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_results.csv"), 
        schema_overrides = schema_map["combine_results"]
    )
    # -------------------------------------
    # Only Keep Needed Variables
    # -------------------------------------
    .select(
        "nfl_id", 
        "ten_yd_split", 
        "forty", 
        "vertical", 
        "broad_jump", 
        "three_cone", 
        "short_shuttle", 
        "ngs_athleticism_score", 
        "ngs_college_production_score", 
        "ngs_final_score"
    )
)

# -----------------------------------------
# Derived Combine Features
# -----------------------------------------

combine_features_df = (
    pl.read_csv(str(SEPARATION_DATA_DIR / "13_DB_Combine_Features_Data.csv"))
    # -------------------------------------
    # Only Keep Needed Variables
    # -------------------------------------
    .select(
        "nfl_id", 
        "accel_score",
        "best_w_decel", 
    )
)

# -----------------------------------------
# Join Combine Data
# -----------------------------------------

combine_df = (
    combine_features_df
    .join(
        combine_results_df, 
        on = "nfl_id", 
        how = "inner"
    )
)

print("Combine Data:")
print(f"    - Dimensions: {combine_df.shape}")
print(f"    - Number of Players: {combine_df.select("nfl_id").n_unique()}")
combine_df

<br><br/>
## Data Cleaning
***

#### Join Player and Combine Data

In [ ]:
# ===================================
# Join Player and Combine Data
# ===================================

analysis_df = (
    players_df
    .join(
        combine_df, 
        on = "nfl_id", 
        how = "left"
    )
)

print("WR and Combine Data:")
print(f"    - Dimensions: {analysis_df.shape}")
print(f"    - Number of Players: {analysis_df.select("nfl_id").n_unique()}")
analysis_df

<br><br/>
## Examine DBs
***

#### Deceleration (`best_w_decel`) Table For CBs

In [ ]:
position_of_interest = "CB"

# ==========================================
# Create Table and Calculate Rank Difference<
# ==========================================

display_df = (
    analysis_df
    .filter(pl.col("nfl_position") == position_of_interest)
    .select([
        "display_name",
        "best_w_decel",
        "draft_round",
        "draft_overall_pick",
        "ngs_athleticism_score",
        "ngs_final_score",
    ])
    .to_pandas()
)

# 1 = best rank
decel_rank = display_df["best_w_decel"].rank(
    method="min",
    ascending=False,
)

ngs_athleticism_rank = display_df["ngs_athleticism_score"].rank(
    method="min",
    ascending=False,
)

# Positive = W-Drill deceleration ranks better than NGS athleticism
display_df["Rank Difference"] = (
    ngs_athleticism_rank - decel_rank
)

# Sort by W-Drill deceleration score
display_df = display_df.sort_values(
    "best_w_decel",
    ascending=False,
    na_position="last",
)

# ==========================================
# Shared Greens Colormap
# ==========================================

greens_cmap = plt.get_cmap("Greens")


def green_color(intensity):
    """Convert a 0-1 intensity to a Greens colormap hex color."""
    intensity = max(0.0, min(1.0, intensity))
    return mcolors.to_hex(greens_cmap(intensity))


def text_color_for_intensity(intensity):
    """Use black on light backgrounds and white on dark backgrounds."""
    rgb = greens_cmap(max(0.0, min(1.0, intensity)))[:3]

    luminance = (
        0.2126 * rgb[0]
        + 0.7152 * rgb[1]
        + 0.0722 * rgb[2]
    )

    return "black" if luminance > 0.45 else "white"


# ==========================================
# Draft Round Color Mapping
# ==========================================

draft_round_order = {
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "UDFA": 8,
}


def draft_round_style(value):
    if pd.isna(value):
        return "background-color: #D3D3D3; color: black;"

    value = str(value).strip().upper()
    rank = draft_round_order.get(value)

    if rank is None:
        return "background-color: #D3D3D3; color: black;"

    # Round 1 = green; UDFA = white
    intensity = (8 - rank) / 7

    return (
        f"background-color: {green_color(intensity)}; "
        f"color: {text_color_for_intensity(intensity)};"
    )


# ==========================================
# Draft Overall Pick Color Mapping
# ==========================================

def draft_pick_style(value):
    if pd.isna(value):
        return "background-color: #D3D3D3; color: black;"

    value = str(value).strip().upper()

    # UDFA = worst = white
    if value == "UDFA":
        intensity = 0
    else:
        try:
            pick = int(value)
        except (ValueError, TypeError):
            return "background-color: #D3D3D3; color: black;"

        if not 1 <= pick <= 257:
            return "background-color: #D3D3D3; color: black;"

        # Pick 1 = green; pick 257 = near-white
        intensity = (258 - pick) / 257

    return (
        f"background-color: {green_color(intensity)}; "
        f"color: {text_color_for_intensity(intensity)};"
    )


# ==========================================
# Symmetric Range for Rank Difference
# ==========================================

rank_max = display_df["Rank Difference"].abs().max()

# Avoid issues if all differences are zero or missing
if pd.isna(rank_max) or rank_max == 0:
    rank_max = 1


# ==========================================
# Format and Style Table
# ==========================================

styled_df = (
    display_df.style

    # Number formatting
    .format({
        "best_w_decel": "{:.2f}",
        "ngs_athleticism_score": "{:.1f}",
        "ngs_final_score": "{:.1f}",
        "Rank Difference": "{:+.0f}",
    }, na_rep="—")

    # W-Drill deceleration
    .background_gradient(
        cmap="Greens",
        subset=["best_w_decel"],
        vmin=analysis_df["best_w_decel"].min(),
        vmax=analysis_df["best_w_decel"].max(),
    )

    # Draft round: Round 1 = green, UDFA = white
    .map(
        draft_round_style,
        subset=["draft_round"],
    )

    # Overall pick: 1 = green, 257 = near-white, UDFA = white
    .map(
        draft_pick_style,
        subset=["draft_overall_pick"],
    )

    # NGS athleticism: 50 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["ngs_athleticism_score"],
        vmin=50,
        vmax=100,
    )

    # NGS final score: 50 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["ngs_final_score"],
        vmin=50,
        vmax=100,
    )

    # Rank Difference:
    # Green = deceleration ranks better than athleticism
    # White = equal rankings
    # Red = deceleration ranks worse than athleticism
    .background_gradient(
        cmap="RdYlGn",
        subset=["Rank Difference"],
        vmin=-rank_max,
        vmax=rank_max,
    )

    # Alignment
    .set_properties(**{
        "text-align": "center",
    })
    .set_properties(
        subset=["display_name"],
        **{"text-align": "left"}
    )

    # Table title
    .set_caption(
        f"{position_of_interest} W-Drill Deceleration, Draft Position, NGS Metrics, and Rank Difference"
    )
)

display(styled_df)

<br><br/>
#### Deceleration (`best_w_decel`) Table For Safeties

In [ ]:
position_of_interest = ["FS", "SS"]
position_name = "FS/SS"

# ==========================================
# Create Table and Calculate Rank Difference
# ==========================================

display_df = (
    analysis_df
    .filter(pl.col("nfl_position").is_in(position_of_interest))
    .select([
        "display_name",
        "best_w_decel",
        "draft_round",
        "draft_overall_pick",
        "ngs_athleticism_score",
        "ngs_final_score",
    ])
    .to_pandas()
)

# 1 = best rank
decel_rank = display_df["best_w_decel"].rank(
    method="min",
    ascending=False,
)

ngs_athleticism_rank = display_df["ngs_athleticism_score"].rank(
    method="min",
    ascending=False,
)

# Positive = W-Drill deceleration ranks better than NGS athleticism
display_df["Rank Difference"] = (
    ngs_athleticism_rank - decel_rank
)

# Sort by W-Drill deceleration score
display_df = display_df.sort_values(
    "best_w_decel",
    ascending=False,
    na_position="last",
)

# ==========================================
# Shared Greens Colormap
# ==========================================

greens_cmap = plt.get_cmap("Greens")


def green_color(intensity):
    """Convert a 0-1 intensity to a Greens colormap hex color."""
    intensity = max(0.0, min(1.0, intensity))
    return mcolors.to_hex(greens_cmap(intensity))


def text_color_for_intensity(intensity):
    """Use black on light backgrounds and white on dark backgrounds."""
    rgb = greens_cmap(max(0.0, min(1.0, intensity)))[:3]

    luminance = (
        0.2126 * rgb[0]
        + 0.7152 * rgb[1]
        + 0.0722 * rgb[2]
    )

    return "black" if luminance > 0.45 else "white"


# ==========================================
# Draft Round Color Mapping
# ==========================================

draft_round_order = {
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "UDFA": 8,
}


def draft_round_style(value):
    if pd.isna(value):
        return "background-color: #D3D3D3; color: black;"

    value = str(value).strip().upper()
    rank = draft_round_order.get(value)

    if rank is None:
        return "background-color: #D3D3D3; color: black;"

    # Round 1 = green; UDFA = white
    intensity = (8 - rank) / 7

    return (
        f"background-color: {green_color(intensity)}; "
        f"color: {text_color_for_intensity(intensity)};"
    )


# ==========================================
# Draft Overall Pick Color Mapping
# ==========================================

def draft_pick_style(value):
    if pd.isna(value):
        return "background-color: #D3D3D3; color: black;"

    value = str(value).strip().upper()

    # UDFA = worst = white
    if value == "UDFA":
        intensity = 0
    else:
        try:
            pick = int(value)
        except (ValueError, TypeError):
            return "background-color: #D3D3D3; color: black;"

        if not 1 <= pick <= 257:
            return "background-color: #D3D3D3; color: black;"

        # Pick 1 = green; pick 257 = near-white
        intensity = (258 - pick) / 257

    return (
        f"background-color: {green_color(intensity)}; "
        f"color: {text_color_for_intensity(intensity)};"
    )


# ==========================================
# Symmetric Range for Rank Difference
# ==========================================

rank_max = display_df["Rank Difference"].abs().max()

# Avoid issues if all differences are zero or missing
if pd.isna(rank_max) or rank_max == 0:
    rank_max = 1


# ==========================================
# Format and Style Table
# ==========================================

styled_df = (
    display_df.style

    # Number formatting
    .format({
        "best_w_decel": "{:.2f}",
        "ngs_athleticism_score": "{:.1f}",
        "ngs_final_score": "{:.1f}",
        "Rank Difference": "{:+.0f}",
    }, na_rep="—")

    # W-Drill deceleration
    .background_gradient(
        cmap="Greens",
        subset=["best_w_decel"],
        vmin=analysis_df["best_w_decel"].min(),
        vmax=analysis_df["best_w_decel"].max(),
    )

    # Draft round: Round 1 = green, UDFA = white
    .map(
        draft_round_style,
        subset=["draft_round"],
    )

    # Overall pick: 1 = green, 257 = near-white, UDFA = white
    .map(
        draft_pick_style,
        subset=["draft_overall_pick"],
    )

    # NGS athleticism: 50 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["ngs_athleticism_score"],
        vmin=50,
        vmax=100,
    )

    # NGS final score: 50 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["ngs_final_score"],
        vmin=50,
        vmax=100,
    )

    # Rank Difference:
    # Green = deceleration ranks better than athleticism
    # White = equal rankings
    # Red = deceleration ranks worse than athleticism
    .background_gradient(
        cmap="RdYlGn",
        subset=["Rank Difference"],
        vmin=-rank_max,
        vmax=rank_max,
    )

    # Alignment
    .set_properties(**{
        "text-align": "center",
    })
    .set_properties(
        subset=["display_name"],
        **{"text-align": "left"}
    )

    # Table title
    .set_caption(
        f"{position_name} W-Drill Deceleration, Draft Position, NGS Metrics, and Rank Difference"
    )
)

display(styled_df)

<br><br/>
#### Deceleration (`best_w_decel`) Table For DBs (CB/FS/SS)

In [ ]:
position_name = "DB"

# ==========================================
# Create Table and Calculate Rank Difference
# ==========================================

display_df = (
    analysis_df
    .select([
        "display_name",
        "best_w_decel",
        "draft_round",
        "draft_overall_pick",
        "ngs_athleticism_score",
        "ngs_final_score",
    ])
    .to_pandas()
)

# 1 = best rank
decel_rank = display_df["best_w_decel"].rank(
    method="min",
    ascending=False,
)

ngs_athleticism_rank = display_df["ngs_athleticism_score"].rank(
    method="min",
    ascending=False,
)

# Positive = W-Drill deceleration ranks better than NGS athleticism
display_df["Rank Difference"] = (
    ngs_athleticism_rank - decel_rank
)

# Sort by W-Drill deceleration score
display_df = display_df.sort_values(
    "best_w_decel",
    ascending=False,
    na_position="last",
)

db_df = display_df

# ==========================================
# Shared Greens Colormap
# ==========================================

greens_cmap = plt.get_cmap("Greens")


def green_color(intensity):
    """Convert a 0-1 intensity to a Greens colormap hex color."""
    intensity = max(0.0, min(1.0, intensity))
    return mcolors.to_hex(greens_cmap(intensity))


def text_color_for_intensity(intensity):
    """Use black on light backgrounds and white on dark backgrounds."""
    rgb = greens_cmap(max(0.0, min(1.0, intensity)))[:3]

    luminance = (
        0.2126 * rgb[0]
        + 0.7152 * rgb[1]
        + 0.0722 * rgb[2]
    )

    return "black" if luminance > 0.45 else "white"


# ==========================================
# Draft Round Color Mapping
# ==========================================

draft_round_order = {
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "UDFA": 8,
}


def draft_round_style(value):
    if pd.isna(value):
        return "background-color: #D3D3D3; color: black;"

    value = str(value).strip().upper()
    rank = draft_round_order.get(value)

    if rank is None:
        return "background-color: #D3D3D3; color: black;"

    # Round 1 = green; UDFA = white
    intensity = (8 - rank) / 7

    return (
        f"background-color: {green_color(intensity)}; "
        f"color: {text_color_for_intensity(intensity)};"
    )


# ==========================================
# Draft Overall Pick Color Mapping
# ==========================================

def draft_pick_style(value):
    if pd.isna(value):
        return "background-color: #D3D3D3; color: black;"

    value = str(value).strip().upper()

    # UDFA = worst = white
    if value == "UDFA":
        intensity = 0
    else:
        try:
            pick = int(value)
        except (ValueError, TypeError):
            return "background-color: #D3D3D3; color: black;"

        if not 1 <= pick <= 257:
            return "background-color: #D3D3D3; color: black;"

        # Pick 1 = green; pick 257 = near-white
        intensity = (258 - pick) / 257

    return (
        f"background-color: {green_color(intensity)}; "
        f"color: {text_color_for_intensity(intensity)};"
    )


# ==========================================
# Symmetric Range for Rank Difference
# ==========================================

rank_max = display_df["Rank Difference"].abs().max()

# Avoid issues if all differences are zero or missing
if pd.isna(rank_max) or rank_max == 0:
    rank_max = 1


# ==========================================
# Format and Style Table
# ==========================================

styled_df = (
    display_df.style

    # Number formatting
    .format({
        "best_w_decel": "{:.2f}",
        "ngs_athleticism_score": "{:.1f}",
        "ngs_final_score": "{:.1f}",
        "Rank Difference": "{:+.0f}",
    }, na_rep="—")

    # W-Drill deceleration
    .background_gradient(
        cmap="Greens",
        subset=["best_w_decel"],
        vmin=analysis_df["best_w_decel"].min(),
        vmax=analysis_df["best_w_decel"].max(),
    )

    # Draft round: Round 1 = green, UDFA = white
    .map(
        draft_round_style,
        subset=["draft_round"],
    )

    # Overall pick: 1 = green, 257 = near-white, UDFA = white
    .map(
        draft_pick_style,
        subset=["draft_overall_pick"],
    )

    # NGS athleticism: 50 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["ngs_athleticism_score"],
        vmin=50,
        vmax=100,
    )

    # NGS final score: 50 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["ngs_final_score"],
        vmin=50,
        vmax=100,
    )

    # Rank Difference:
    # Green = deceleration ranks better than athleticism
    # White = equal rankings
    # Red = deceleration ranks worse than athleticism
    .background_gradient(
        cmap="RdYlGn",
        subset=["Rank Difference"],
        vmin=-rank_max,
        vmax=rank_max,
    )

    # Alignment
    .set_properties(**{
        "text-align": "center",
    })
    .set_properties(
        subset=["display_name"],
        **{"text-align": "left"}
    )

    # Table title
    .set_caption(
        f"{position_name} W-Drill Deceleration, Draft Position, NGS Metrics, and Rank Difference"
    )
)

display(styled_df)

In [ ]:
# ===============================================
# Save Decel Comparison Table To CSV
# ===============================================

db_df.pipe(pl.from_pandas).write_csv(str(SEPARATION_DATA_DIR / "32_DB_Decel_Comparison_Table.csv"))

<br><br/>
## Deceleration vs Acceleration Among DBs
***

In [ ]:
accel_decel_df = (
    analysis_df
    .select(
        "display_name", 
        "nfl_id", 
        "nfl_position", 
        "accel_score", 
        "best_w_decel"
    )
)

accel_decel_df

<br><br/>
#### Comparing CBs

In [ ]:
# ============================================
# Deceleration vs Accleration In Players
# ============================================

position_of_interest = ["CB"]
position_name = "CBs"

fig = px.scatter(
    data_frame = accel_decel_df.filter(pl.col("nfl_position").is_in(position_of_interest)), 
    x = "accel_score", 
    y = "best_w_decel", 
    hover_data = "display_name",
    title = f"Acceleration and Deceleration Among {position_name}", 
    labels = {
        "accel_score": "Acceleration Score", 
        "best_w_decel": "Deceleration"
    },
    template = "simple_white"
)

# Get axis ranges
x_min, x_max = accel_decel_df["accel_score"].min(), accel_decel_df["accel_score"].max()
y_min, y_max = accel_decel_df["best_w_decel"].min(), accel_decel_df["best_w_decel"].max()

x_mid = (x_min + x_max) / 2
y_mid = (y_min + y_max) / 2

# Bottom-left quadrant: red
fig.add_shape(
    type="rect",
    x0=x_min, x1=x_mid,
    y0=y_min, y1=y_mid,
    fillcolor="red",
    opacity=0.15,
    line_width=0,
    layer="below"
)

# Top-left quadrant: green
fig.add_shape(
    type="rect",
    x0=x_mid, x1=x_max,
    y0=y_mid, y1=y_max,
    fillcolor="green",
    opacity=0.15,
    line_width=0,
    layer="below"
)

fig.show()

<br><br/>
#### Comparing Safeties

In [ ]:
# ============================================
# Deceleration vs Accleration In Players
# ============================================

position_of_interest = ["FS", "SS"]
position_name = "Safeties"

fig = px.scatter(
    data_frame = accel_decel_df.filter(pl.col("nfl_position").is_in(position_of_interest)), 
    x = "accel_score", 
    y = "best_w_decel", 
    hover_data = "display_name",
    title = f"Acceleration and Deceleration Among {position_name}", 
    labels = {
        "accel_score": "Acceleration Score", 
        "best_w_decel": "Deceleration"
    },
    template = "simple_white"
)

# Get axis ranges
x_min, x_max = accel_decel_df["accel_score"].min(), accel_decel_df["accel_score"].max()
y_min, y_max = accel_decel_df["best_w_decel"].min(), accel_decel_df["best_w_decel"].max()

x_mid = (x_min + x_max) / 2
y_mid = (y_min + y_max) / 2

# Bottom-left quadrant: red
fig.add_shape(
    type="rect",
    x0=x_min, x1=x_mid,
    y0=y_min, y1=y_mid,
    fillcolor="red",
    opacity=0.15,
    line_width=0,
    layer="below"
)

# Top-left quadrant: green
fig.add_shape(
    type="rect",
    x0=x_mid, x1=x_max,
    y0=y_mid, y1=y_max,
    fillcolor="green",
    opacity=0.15,
    line_width=0,
    layer="below"
)

fig.show()

<br><br/>
#### Comparing DBs

In [ ]:
# ============================================
# Deceleration vs Accleration In Players
# ============================================

position_of_interest = ["CB", "FS", "SS"]
position_name = "DBs"

fig = px.scatter(
    data_frame = accel_decel_df.filter(pl.col("nfl_position").is_in(position_of_interest)), 
    x = "accel_score", 
    y = "best_w_decel", 
    hover_data = "display_name",
    title = f"Acceleration and Deceleration Among {position_name}", 
    labels = {
        "accel_score": "Acceleration Score", 
        "best_w_decel": "Deceleration"
    },
    template = "simple_white"
)

# Get axis ranges
x_min, x_max = accel_decel_df["accel_score"].min(), accel_decel_df["accel_score"].max()
y_min, y_max = accel_decel_df["best_w_decel"].min(), accel_decel_df["best_w_decel"].max()

x_mid = (x_min + x_max) / 2
y_mid = (y_min + y_max) / 2

# Bottom-left quadrant: red
fig.add_shape(
    type="rect",
    x0=x_min, x1=x_mid,
    y0=y_min, y1=y_mid,
    fillcolor="red",
    opacity=0.15,
    line_width=0,
    layer="below"
)

# Top-left quadrant: green
fig.add_shape(
    type="rect",
    x0=x_mid, x1=x_max,
    y0=y_mid, y1=y_max,
    fillcolor="green",
    opacity=0.15,
    line_width=0,
    layer="below"
)

fig.show()